In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from go_ml.eval_utils import filter_annot_df
from go_ml.eval_utils import (load_msa_dict, gen_bert_mat, get_bert_entropy, 
                              gen_annot_mat, gen_seq_len_mask, mean_reciprocal_rank, 
                              mean_reciprocal_rank_mat, mean_auc, top_30_score, roc_average)

data_root = '/home/andrew/GO_interp/go_ml/gen_datasets/datasets'
# csa_df = filter_annot_df(pd.read_csv(f'{data_root}/csa_annot.csv', sep='\t'))
# llps_df = filter_annot_df(pd.read_csv(f'{data_root}/llps_dataset.csv', sep='\t'))
# elms_df = filter_annot_df(pd.read_csv(f'{data_root}/elms_dataset.csv', sep='\t'))
# dataset_labels = ['csa', 'llps', 'elms', 'biolip', 
#                   'ip_repeat', 'ip_homologous_superfamily', 'ip_domain', 'ip_binding_site', 'ip_active_site']
dataset_labels = ['bidomain']
dataset_dfs = [filter_annot_df(pd.read_csv(f'{data_root}/{label}_dataset.csv', sep='\t')) for label in dataset_labels]

In [2]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from esm.models.esmc import ESMC
device = torch.device('cuda:0')
model = ESMC.from_pretrained("esmc_600m").to(device) # or "cpu"
model.eval()
vi = {i: a for a, i in model.tokenizer.get_vocab().items()}
AA_str = [vi[i] for i in range(4, 24)]
from go_ml.masking import mask_perc, get_logits_esmc

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [21]:
from esm.utils.constants.esm3 import (
    SEQUENCE_MASK_TOKEN,
)
from esm.sdk.api import ESMProtein, LogitsConfig

In [4]:
from tqdm.notebook import tqdm
import pickle
import os

for ds_label, annot_df in zip(dataset_labels, dataset_dfs):
    # if ds_label not in ['ip_repeat', 'ip_homologous_superfamily', 'ip_family', 'ip_domain']:
    #     continue
    fp = f'/home/andrew/GO_interp/go_ml/dataset_eval/eval_files/{ds_label}/esmc.pkl'
    if(os.path.exists(fp)):
        print(f'Skipping {ds_label}, already done')
        continue
    df_logits = {}
    for i, (seq_id, seq) in tqdm(enumerate(zip(annot_df['UniprotID'], annot_df['Sequence'])), 
                                 total=len(annot_df), desc=f'Processing {ds_label}'):
        df_logits[seq_id] = get_logits_esmc(seq, model, batch_size=24, mask_func=mask_perc).float()
    bert_map = {k: v[:, 4:24] for k, v in df_logits.items()}
    bert_map = {k: v / (v.sum(dim=1, keepdims=True) + 1e-10) for k, v in bert_map.items()}
    bert_map = {k: v.numpy() for k, v in bert_map.items()}

    annot_mat = gen_annot_mat(annot_df['AnnotatedIndices'], [len(s) for s in annot_df['Sequence']])
    seq_len_mask = gen_seq_len_mask(annot_df['Sequence'])
    bert_mat = gen_bert_mat(annot_df['UniprotID'], bert_map, max_len=850)
    bert_entropy = get_bert_entropy(bert_mat, seq_len_mask)
    with open(fp, 'wb') as f:
            pickle.dump({'UniprotID': annot_df['UniprotID'], 'bert_mat': bert_mat, 
                        'seq_len_mask': seq_len_mask, 'bert_entropy': bert_entropy}, f)
    print(f'DS label: {ds_label}')

Processing bidomain:   0%|          | 0/771 [00:00<?, ?it/s]

DS label: bidomain


In [ ]:
# from tqdm.notebook import tqdm
# import pickle

# ds_labels = ['csa', 'llps', 'elms']
# for ds_label, annot_df in zip(ds_labels, [csa_df, llps_df, elms_df]):
#     df_logits = {}
#     for i, (seq_id, seq) in tqdm(enumerate(zip(annot_df['UniprotID'], annot_df['Sequence'])), 
#                                  total=len(annot_df), desc=f'Processing {ds_label}'):
#         df_logits[seq_id] = get_logits_esmc(seq, model, batch_size=8, mask_func=mask_perc).float()
#     bert_map = {k: v[:, 4:24] for k, v in df_logits.items()}
#     bert_map = {k: v / (v.sum(dim=1, keepdims=True) + 1e-10) for k, v in bert_map.items()}
#     bert_map = {k: v.numpy() for k, v in bert_map.items()}

#     annot_mat = gen_annot_mat(annot_df['AnnotatedIndices'], [len(s) for s in annot_df['Sequence']])
#     seq_len_mask = gen_seq_len_mask(annot_df['Sequence'])
#     bert_mat = gen_bert_mat(annot_df['UniprotID'], bert_map, max_len=850)
#     bert_entropy = get_bert_entropy(bert_mat, seq_len_mask)
#     with open(f'eval_files/{ds_label}_esmc.pkl', 'wb') as f:
#             pickle.dump({'UniprotID': annot_df['UniprotID'], 'bert_mat': bert_mat, 
#                         'seq_len_mask': seq_len_mask, 'bert_entropy': bert_entropy}, f)
#     print(f'DS label: {ds_label}')

Processing elms:   0%|          | 0/229 [00:00<?, ?it/s]

DS label: elms
